# 01 Inspect RAW

- apre il layer RAW reale del toolkit
- legge `manifest.json`, `metadata.json` e `raw_validation.json`
- prova a mostrare un sample del file primario dichiarato nel manifest

In [ ]:
from pathlib import Path
import json
import shutil
import subprocess
import duckdb
import yaml

ROOT = Path('.').resolve()
DATASET_YML = (ROOT / 'dataset.yml').resolve() if (ROOT / 'dataset.yml').exists() else (ROOT / '..' / 'dataset.yml').resolve()
CFG = yaml.safe_load(DATASET_YML.read_text(encoding='utf-8'))
DATASET = CFG['dataset']['name']
YEARS = CFG['dataset']['years']
YEAR_INDEX = 0
YEAR = YEARS[YEAR_INDEX] if YEARS and 0 <= YEAR_INDEX < len(YEARS) else YEARS[0]
CLI_PREFIX = ['toolkit'] if shutil.which('toolkit') else ['py', '-m', 'toolkit.cli.app']
INSPECT_CMD = CLI_PREFIX + ['inspect', 'paths', '--config', str(DATASET_YML), '--year', str(YEAR), '--json']
INSPECT = json.loads(subprocess.run(INSPECT_CMD, capture_output=True, text=True, check=True).stdout)
RAW_DIR = Path(INSPECT['paths']['raw']['dir'])
MANIFEST_PATH = Path(INSPECT['paths']['raw']['manifest'])
METADATA_PATH = Path(INSPECT['paths']['raw']['metadata'])
VALIDATION_PATH = Path(INSPECT['paths']['raw']['validation'])
PROFILE_DIR = RAW_DIR / '_profile'

{
    'YEARS': YEARS,
    'YEAR_INDEX': YEAR_INDEX,
    'RAW_DIR': str(RAW_DIR),
    'INSPECT_CMD': INSPECT_CMD,
    'MANIFEST_EXISTS': MANIFEST_PATH.exists(),
    'METADATA_EXISTS': METADATA_PATH.exists(),
    'VALIDATION_EXISTS': VALIDATION_PATH.exists(),
}

In [ ]:
manifest = json.loads(MANIFEST_PATH.read_text(encoding='utf-8')) if MANIFEST_PATH.exists() else {}
metadata = json.loads(METADATA_PATH.read_text(encoding='utf-8')) if METADATA_PATH.exists() else {}
validation = json.loads(VALIDATION_PATH.read_text(encoding='utf-8')) if VALIDATION_PATH.exists() else {}

display(manifest)
display(metadata)
display(validation)

In [ ]:
primary_rel = manifest.get('primary_output_file') if manifest else None
primary_path = (RAW_DIR / primary_rel).resolve() if primary_rel else None
primary_path

In [ ]:
if primary_path and primary_path.exists():
    suffix = primary_path.suffix.lower()
    if suffix == '.csv':
        con = duckdb.connect()
        preview = con.execute(
            f"SELECT * FROM read_csv_auto('{primary_path.as_posix()}', SAMPLE_SIZE=-1) LIMIT 20"
        ).df()
        display(preview)
    elif suffix == '.parquet':
        con = duckdb.connect()
        preview = con.execute(f"SELECT * FROM read_parquet('{primary_path.as_posix()}') LIMIT 20").df()
        display(preview)
    else:
        print({'primary_output_file': str(primary_path), 'bytes': primary_path.stat().st_size, 'suffix': suffix})
else:
    print('Primary output file not found. Inspect manifest manually.')

if PROFILE_DIR.exists():
    display(sorted(path.name for path in PROFILE_DIR.iterdir()))
else:
    print('No _profile directory found for this RAW run.')